# DLP Detection Model — CatBoost Training Pipeline


In [0]:
%pip install catboost shap numpy==1.26.4 --quiet

In [0]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix, precision_recall_curve, f1_score
import shap, pickle, os

## 1. Load Data

In [0]:
df_features_clean = spark.table("workspace.default.dlp_features_clean")
df = df_features_clean.toPandas()

print(f"Total rows: {len(df)}")
print(df["Risk_Label"].value_counts())

## 2. Define Feature Columns

In [0]:
LABEL_COL = "Risk_Label"

# CatBoost handles categoricals natively — no encoding needed
CATEGORICAL_COLS = [
    "ActionType",
    "file_extension",
    "Position",
]

NUMERICAL_COLS = [
    "is_high_risk_extension",
    "file_name_has_sensitive_keyword",
    "double_extension_check",
    "is_risky_target_domain",
    "is_first_time_domain",
    "is_after_hours",
    "day_of_week",
    "is_monday_or_friday",
    "is_high_risk_position",
    "user_upload_count_24h",
    "user_unique_domains_7d",
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS

CAT_FEATURE_INDICES = [FEATURE_COLS.index(c) for c in CATEGORICAL_COLS]

X = df[FEATURE_COLS].copy()
y = df[LABEL_COL]

for c in CATEGORICAL_COLS:
    X[c] = X[c].astype(str)

print(f"Features: {len(FEATURE_COLS)}")
print(f"X shape: {X.shape}")

## 3. Train / Validation / Test Split
CatBoost early stopping requires a separate validation set during training,
so we split into 3: train (64%), validation (16%), test (20%).

In [0]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# split train into train + validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

print(f"Train : {len(X_train)}")
print(f"Val   : {len(X_val)}")
print(f"Test  : {len(X_test)}")

## 4. Train with Early Stopping
CatBoost evaluates AUC on the validation set after each tree.
Training stops automatically when validation AUC stops improving
for `early_stopping_rounds` consecutive iterations — prevents overfitting.

In [0]:
model = CatBoostClassifier(
    iterations=500,             
    learning_rate=0.1,
    depth=3,                    
    eval_metric="AUC",          
    early_stopping_rounds=50,   
    random_seed=42,
    verbose=50                  
)

model.fit(
    X_train, y_train,
    cat_features=CAT_FEATURE_INDICES,
    eval_set=(X_val, y_val),    
    plot=True                  
)

print(f"\nBest iteration: {model.best_iteration_}")

## 5. Evaluate 


In [0]:
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print(f"AUC-ROC  : {roc_auc_score(y_test, y_pred_prob):.4f}")
print(f"AUC-PR   : {average_precision_score(y_test, y_pred_prob):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 6. Feature Importance

In [0]:
importance_df = (
    pd.DataFrame({
        "feature":    FEATURE_COLS,
        "importance": model.get_feature_importance()
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print(importance_df.to_string(index=False))

## 7. SHAP Explainability

In [0]:
action_id = 163

df_full = df.copy()
for c in CATEGORICAL_COLS:
    df_full[c] = df_full[c].astype(str)

df_full["predicted"]      = model.predict(df_full[FEATURE_COLS])
df_full["predicted_prob"] = model.predict_proba(df_full[FEATURE_COLS])[:, 1]

row_idx = df_full[df_full["Action_ID"] == action_id].index[0]
row     = df_full.loc[row_idx]

print(f"Action_ID          : {row['Action_ID']}")
print(f"ObjectName         : {row['ObjectName']}")
print(f"Target_Domain      : {row['Target_Domain']}")
print(f"AccountDisplayName : {row['AccountDisplayName']}")
print(f"Position           : {row['Position']}")
print(f"Actual label       : {row['Risk_Label']}")
print(f"Predicted          : {row['predicted']}")
print(f"Probability        : {float(row['predicted_prob']):.4f}")
print()

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(df_full.loc[[row_idx], FEATURE_COLS])

shap_df = (
    pd.DataFrame({
        "feature":    FEATURE_COLS,
        "value":      df_full.loc[row_idx, FEATURE_COLS].values,
        "shap_value": shap_values[0]
    })
    .sort_values("shap_value", ascending=False)
    .reset_index(drop=True)
)
print(shap_df.to_string(index=False))

## 8. False Negatives Analysis

In [0]:
false_negatives = df_full.loc[X_test.index][
    (df_full.loc[X_test.index]["Risk_Label"] == 1) &
    (df_full.loc[X_test.index]["predicted"] == 0)
]

print(f"False negatives: {len(false_negatives)}")
print(
    false_negatives[[
        "Action_ID", "ObjectName", "Target_Domain",
        "AccountDisplayName", "Position", "predicted_prob"
    ]]
    .sort_values("predicted_prob", ascending=False)
    .to_string(index=False)
)

## 9. Save Model

In [0]:
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = os.path.dirname(notebook_path)

model.save_model(f"/Workspace{repo_root}/dlp_catboost_v1.cbm")
print("Model saved successfully.")